# EPL448 – Deliverable 3: Predictive Model Development & Performance Evaluation
## CERN Electron Collision Dataset – Team 2
**Members:** Varnavas Tryfonos, Thrasos Sazeidis, Andreas Evagorou  
**Dataset:** [CERN Electron Collision Data (Kaggle)](https://www.kaggle.com/datasets/fedesoriano/cern-electron-collision-data)

---
## 0. Setup & Imports

In [ ]:
import sys
from pathlib import Path

# Allow notebooks/ directory to import from src/
sys.path.insert(0, str(Path('..').resolve()))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')   # non-interactive backend for saving figures
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Targeted warning suppression – do NOT silence all warnings globally
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning, module='sklearn')
warnings.filterwarnings('ignore', message='.*max_iter.*')

# Sklearn
from sklearn.model_selection import (
    train_test_split, cross_validate, KFold, GridSearchCV
)
from sklearn.compose import TransformedTargetRegressor

# ── Project modules ────────────────────────────────────────────────────────
from src.features import LOG_FEATURES, add_engineered_features, build_v1, build_v2, build_v3, build_v4
from src.evaluation import compute_metrics, CV_SCORING
from src.models import build_pipeline, PARAM_GRIDS, N_SVR_SCREEN, N_TOP_MODELS, N_TOP_DATASETS, RANDOM_STATE
from src.validation import validate_raw, validate_clean

# ── Plot defaults ──────────────────────────────────────────────────────────
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (12, 5)
sns.set_style('whitegrid')

# Output directory for generated figures
OUTPUTS_DIR = Path('../outputs')
OUTPUTS_DIR.mkdir(exist_ok=True)

print('All imports successful.')


---
## 1. Load Dataset & Reproduce Preprocessing from Deliverable 2

In [ ]:
# Load data – place dielectron.csv in the data/ directory (see data/README.md)
DATA_PATH = Path('../data/dielectron.csv')
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}.\n"
        "Download dielectron.csv and place it in the data/ directory.\n"
        "See data/README.md for instructions."
    )

df = pd.read_csv(DATA_PATH)

# ── Fix L1: strip trailing whitespace from column names ───────────────────
df.columns = df.columns.str.strip()

print(f'Dataset loaded: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Columns: {list(df.columns)}')

# ── Data validation (H5) ──────────────────────────────────────────────────
validate_raw(df)
print('Raw data validation passed.')

# Drop non-predictive identifier columns
df_clean = df.drop(columns=['Run', 'Event'], errors='ignore').copy()
df_clean = df_clean.dropna(subset=['M']).reset_index(drop=True)

validate_clean(df_clean)
print(f'Working dataset: {df_clean.shape[0]:,} rows x {df_clean.shape[1]} columns')
print('Clean data validation passed.')


### 1.1 Feature Engineering Functions (from Deliverable 2)

In [ ]:
# Feature engineering functions are defined in src/features.py
# Imported above: add_engineered_features, LOG_FEATURES, build_v1/v2/v3/v4
#
# Key fix (M6): delta_phi is now vectorised with np.minimum (no .apply/lambda)
print('Feature engineering module loaded from src/features.py')
print(f'LOG_FEATURES: {LOG_FEATURES}')


### 1.2 Revisions from Deliverable 2

**Key revision based on feedback:** In Deliverable 2, all dataset versions were scaled using `StandardScaler`.
The professor noted that tree-based models (Random Forest, XGBoost) are **scale-invariant** and do not benefit from scaling.

**Changes made:**
1. Scaling is no longer pre-applied to dataset versions. Instead, it is included as a **pipeline step** so that:
   - It is only applied when the model requires it (KNN, SVR).
   - It is fitted only on the training folds during cross-validation (no data leakage).
2. All preprocessing (log-transform, scaling, PCA) is now handled **inside scikit-learn Pipelines** fed to `GridSearchCV`.
3. Model evaluation in Deliverable 2 was preliminary and is now replaced with the proper workflow: default models → screening → selection → pipeline tuning.

### 1.3 Build Dataset Versions (Unscaled)

We prepare the raw feature matrices for each version. **Scaling is NOT applied here** — it will be
handled inside pipelines where needed.

In [ ]:
# ── Targets ────────────────────────────────────────────────────────────────
y = df_clean['M'].values

# ── Build feature matrices using src/features.py builders ─────────────────
X_v1 = build_v1(df_clean)
X_v2 = build_v2(df_clean)
X_v3 = build_v3(df_clean)  # same features as V2; log target applied via TransformedTargetRegressor
X_v4 = build_v4(df_clean)

# ── Stratified train/test split ────────────────────────────────────────────
# Use pd.qcut with q=10 bins; duplicates='drop' handles ties in the mass dist.
mass_bins = pd.qcut(y, q=10, labels=False, duplicates='drop')

datasets_raw = {}
for name, X_df, y_arr in [
    ('V1', X_v1, y),
    ('V2', X_v2, y),
    ('V3', X_v3, y),        # raw target; pipeline wraps with TransformedTargetRegressor
    ('V4', X_v4, y),
]:
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_df.values, y_arr, test_size=0.2,
        random_state=RANDOM_STATE, stratify=mass_bins
    )
    datasets_raw[name] = {
        'X_train':       X_tr,
        'X_test':        X_te,
        'y_train':       y_tr,
        'y_test':        y_te,
        'feature_names': list(X_df.columns),
        'log_target':    (name == 'V3'),
    }
    print(f'{name}: train={X_tr.shape}, test={X_te.shape}, log_target={name == "V3"}')

print(f'\nFeature counts: V1={X_v1.shape[1]}, V2={X_v2.shape[1]}, '
      f'V3={X_v3.shape[1]}, V4={X_v4.shape[1]}')


---
## 2. Model Evaluation Metrics

Since this is a **regression** problem (predicting invariant mass M in GeV), we use the following metrics:

| Metric | Formula | Why we use it |
|--------|---------|---------------|
| **RMSE** | $\sqrt{\frac{1}{n}\sum(y_i - \hat{y}_i)^2}$ | Penalises large errors more heavily. Important because errors on Z boson events (~91 GeV) are physically significant. Same units as target (GeV). |
| **MAE** | $\frac{1}{n}\sum|y_i - \hat{y}_i|$ | Robust to outliers. Gives a sense of typical prediction error in GeV. |
| **R²** | $1 - \frac{\sum(y_i - \hat{y}_i)^2}{\sum(y_i - \bar{y})^2}$ | Proportion of variance explained. Easy to compare across studies. 1.0 = perfect, 0.0 = baseline (predict mean). |
| **MAPE** | $\frac{100}{n}\sum\left|\frac{y_i - \hat{y}_i}{y_i}\right|$ | Relative error — important because M spans a wide range (2–110 GeV). A 5 GeV error means very different things at M=3 GeV vs M=91 GeV. |

**Primary metric for model selection:** R² (used for cross-validation ranking).  
**Secondary metrics:** RMSE and MAPE (reported for interpretation).

In [ ]:
# compute_metrics and CV_SCORING are defined in src/evaluation.py and imported above.
# cv_scoring alias kept for backward compatibility within this notebook.
cv_scoring = CV_SCORING
print('Evaluation metrics loaded from src/evaluation.py')


---
## 3. Initial Model Experimentation – Default Hyperparameters

We train all four models with **default hyperparameters** on all dataset versions using
**5-fold cross-validation**. The goal is to identify which model × dataset combinations
perform best before investing in hyperparameter tuning.

**Important:** For models that require scaling (KNN, SVR), we wrap them in a `Pipeline`
with `StandardScaler` so scaling is fitted only on training folds. Tree-based models
(Random Forest, XGBoost) are used without scaling.

In [ ]:
# build_pipeline is defined in src/models.py and imported above.
# It handles the scaler/no-scaler logic for KNN, SVR, RF, XGB.

def get_default_models():
    """Return dict of model_name -> pipeline with default hyperparameters."""
    return {name: build_pipeline(name) for name in ('KNN', 'SVR', 'RF', 'XGB')}

print('Default models (using src/models.build_pipeline):')
for name, m in get_default_models().items():
    steps = ' → '.join(s[0] for s in m.steps)
    print(f'  {name}: {steps}')


In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
all_cv_results = []

for ver_name, ver_data in datasets_raw.items():
    X_tr = ver_data['X_train']
    y_tr = ver_data['y_train']

    models = get_default_models()

    for model_name, model in models.items():
        # SVR is O(n²–n³) in training set size.  We draw a fixed subsample
        # before cross_validate for speed.  The subsample is drawn with a
        # fixed seed so results are reproducible.  Limitation: all 5 folds
        # see the same 20K subset rather than independent draws; acceptable
        # for screening – the full data is used in GridSearchCV tuning below.
        if model_name == 'SVR':
            rng = np.random.default_rng(RANDOM_STATE)
            n   = min(N_SVR_SCREEN, len(X_tr))
            idx = rng.choice(len(X_tr), n, replace=False)
            X_cv, y_cv = X_tr[idx], y_tr[idx]
        else:
            X_cv, y_cv = X_tr, y_tr

        # V3 uses a log-transformed target; wrap in TransformedTargetRegressor
        # so CV scores (R², RMSE, MAE) are computed on the original GeV scale.
        cv_estimator = model
        if ver_data['log_target']:
            cv_estimator = TransformedTargetRegressor(
                regressor=model, func=np.log, inverse_func=np.exp
            )
        cv_res = cross_validate(
            cv_estimator, X_cv, y_cv, cv=kf,
            scoring=cv_scoring, n_jobs=-1,
            return_train_score=False,
        )

        result = {
            'Model':     model_name,
            'Dataset':   ver_name,
            'R2_mean':   cv_res['test_R2'].mean(),
            'R2_std':    cv_res['test_R2'].std(),
            'RMSE_mean': -cv_res['test_RMSE'].mean(),
            'RMSE_std':  cv_res['test_RMSE'].std(),
            'MAE_mean':  -cv_res['test_MAE'].mean(),
            'MAE_std':   cv_res['test_MAE'].std(),
        }
        all_cv_results.append(result)
        print(f'{model_name:4s} x {ver_name}: '
              f'R²={result["R2_mean"]:.4f} (±{result["R2_std"]:.4f})  '
              f'RMSE={result["RMSE_mean"]:.3f}  MAE={result["MAE_mean"]:.3f}')
    print()

cv_df = pd.DataFrame(all_cv_results)
print('Screening complete.')


### 3.1 Screening Results Table

In [9]:
# Pivot table: rows = models, columns = dataset versions, values = R²
pivot_r2 = cv_df.pivot(index='Model', columns='Dataset', values='R2_mean')
pivot_r2 = pivot_r2[['V1', 'V2', 'V3', 'V4']]  # order columns
pivot_r2 = pivot_r2.loc[['KNN', 'SVR', 'RF', 'XGB']]  # order rows

print('=== Cross-Validated R² Scores (Default Hyperparameters) ===')
print()
display(pivot_r2.style.format('{:.4f}').highlight_max(axis=None, color='lightgreen'))

# Also show RMSE
pivot_rmse = cv_df.pivot(index='Model', columns='Dataset', values='RMSE_mean')
pivot_rmse = pivot_rmse[['V1', 'V2', 'V3', 'V4']]
pivot_rmse = pivot_rmse.loc[['KNN', 'SVR', 'RF', 'XGB']]

print('\n=== Cross-Validated RMSE (Default Hyperparameters) ===')
print()
display(pivot_rmse.style.format('{:.3f}').highlight_min(axis=None, color='lightgreen'))

=== Cross-Validated R² Scores (Default Hyperparameters) ===



Dataset,V1,V2,V3,V4
Model,,,,
KNN,0.9038,0.9482,0.9210,0.9553
SVR,0.8975,0.9344,0.9180,0.9544
RF,0.9456,0.9456,0.8583,0.9963
XGB,0.9773,0.9773,0.9354,0.9952



=== Cross-Validated RMSE (Default Hyperparameters) ===



Dataset,V1,V2,V3,V4
Model,,,,
KNN,7.833,5.746,0.253,5.337
SVR,8.088,6.468,0.259,5.389
RF,5.889,5.887,0.339,1.533
XGB,3.803,3.801,0.229,1.746


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
model_colors = {'KNN': 'steelblue', 'SVR': 'darkorange', 'RF': 'seagreen', 'XGB': 'crimson'}

ax = axes[0]
bar_data = cv_df.copy()
bar_data['label'] = bar_data['Model'] + ' / ' + bar_data['Dataset']
bar_data = bar_data.sort_values('R2_mean', ascending=True)
colors = [model_colors[m] for m in bar_data['Model']]
ax.barh(bar_data['label'], bar_data['R2_mean'], xerr=bar_data['R2_std'],
        color=colors, edgecolor='none', capsize=3)
ax.set_xlabel('R² (5-fold CV)', fontsize=11)
ax.set_title('Model × Dataset Screening – R²', fontsize=13)
ax.axvline(0, color='grey', linewidth=0.5)

ax = axes[1]
bar_data2 = cv_df.sort_values('RMSE_mean', ascending=False)
colors2 = [model_colors[m] for m in bar_data2['Model']]
ax.barh(bar_data2['Model'] + ' / ' + bar_data2['Dataset'], bar_data2['RMSE_mean'],
        xerr=bar_data2['RMSE_std'], color=colors2, edgecolor='none', capsize=3)
ax.set_xlabel('RMSE in GeV (5-fold CV)', fontsize=11)
ax.set_title('Model × Dataset Screening – RMSE', fontsize=13)

from matplotlib.patches import Patch
legend_patches = [Patch(facecolor=c, label=m) for m, c in model_colors.items()]
axes[0].legend(handles=legend_patches, loc='lower right', fontsize=9)

plt.tight_layout()
fig_path = OUTPUTS_DIR / 'fig_screening_results.png'
plt.savefig(fig_path, bbox_inches='tight')
plt.show()
print(f'Figure saved to {fig_path}')


---
## 4. Selection of Best-Performing Models & Dataset Versions

Based on the screening results above, we select the **top 2–3 models** and **top 2 dataset versions**
for hyperparameter tuning.

In [ ]:
# Identify top model–dataset combinations by R²
top_combos = cv_df.nlargest(6, 'R2_mean')[['Model', 'Dataset', 'R2_mean', 'RMSE_mean']]
print('Top 6 model × dataset combinations by R²:')
print(top_combos.to_string(index=False))

model_avg = cv_df.groupby('Model')['R2_mean'].mean().sort_values(ascending=False)
print('\nAverage R² by model (across all datasets):')
print(model_avg.to_string())

dataset_avg = cv_df.groupby('Dataset')['R2_mean'].mean().sort_values(ascending=False)
print('\nAverage R² by dataset (across all models):')
print(dataset_avg.to_string())

# ── Automated selection (M3 fix) ──────────────────────────────────────────
# Top N models and datasets are selected by mean CV R² – no hardcoding.
SELECTED_MODELS   = model_avg.nlargest(N_TOP_MODELS).index.tolist()
SELECTED_DATASETS = dataset_avg.nlargest(N_TOP_DATASETS).index.tolist()

print(f'\n>>> Selected models for tuning:   {SELECTED_MODELS}')
print(f'>>> Selected datasets for tuning: {SELECTED_DATASETS}')


---
## 5. Pipeline Definition

We construct **scikit-learn Pipelines** that integrate preprocessing with the model.
This ensures that:
- Preprocessing is fitted **only on training folds** during cross-validation.
- `GridSearchCV` handles train/validation splitting internally.
- There is **no data leakage** from test data into preprocessing.

For **tree-based models** (RF, XGBoost): the pipeline is just the model (no scaling needed).  
For **distance/kernel-based models** (KNN, SVR): the pipeline includes `StandardScaler` before the model.

In [ ]:
# build_pipeline is defined in src/models.py (imported above).
# Show pipeline structures for selected models.
for name in SELECTED_MODELS:
    pipe = build_pipeline(name)
    print(f'{name} pipeline: {" → ".join(s[0] for s in pipe.steps)}')
    print()


---
## 6. Hyperparameter Tuning with GridSearchCV

For each selected model × dataset combination, we define a hyperparameter grid and
run `GridSearchCV` with 5-fold cross-validation. The pipeline ensures preprocessing
is applied correctly within each fold.

In [ ]:
# PARAM_GRIDS is defined in src/models.py (imported above).
# Print grid sizes for selected models.
for name in SELECTED_MODELS:
    grid = PARAM_GRIDS[name]
    n_combos = 1
    for v in grid.values():
        n_combos *= len(v)
    print(f'{name}: {n_combos} combinations × 5 folds = {n_combos * 5} fits')


In [ ]:
grid_results = {}

for model_name in SELECTED_MODELS:
    for dataset_name in SELECTED_DATASETS:
        key = f'{model_name}_{dataset_name}'
        print(f'\n{"="*60}')
        print(f'Tuning: {model_name} on {dataset_name}')
        print(f'{"="*60}')

        ver   = datasets_raw[dataset_name]
        X_tr  = ver['X_train']
        y_tr  = ver['y_train']
        pipe  = build_pipeline(model_name)

        # V3: wrap pipeline with TransformedTargetRegressor so GridSearchCV
        # scores on the original GeV scale; prefix PARAM_GRIDS keys accordingly.
        if ver['log_target']:
            search_est = TransformedTargetRegressor(
                regressor=pipe, func=np.log, inverse_func=np.exp
            )
            param_grid = {f'regressor__{k}': v for k, v in PARAM_GRIDS[model_name].items()}
        else:
            search_est = pipe
            param_grid = PARAM_GRIDS[model_name]

        # SVR: subsample for feasible tuning time
        if model_name == 'SVR':
            rng = np.random.default_rng(RANDOM_STATE)
            n   = min(N_SVR_SCREEN, len(X_tr))
            idx = rng.choice(len(X_tr), n, replace=False)
            X_search, y_search = X_tr[idx], y_tr[idx]
        else:
            X_search, y_search = X_tr, y_tr

        gs = GridSearchCV(
            estimator=search_est,
            param_grid=param_grid,
            cv=KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
            scoring='r2',
            n_jobs=-1,
            verbose=1,
            return_train_score=True,
        )
        gs.fit(X_search, y_search)

        grid_results[key] = {
            'grid_search':   gs,
            'best_params':   gs.best_params_,
            'best_cv_r2':    gs.best_score_,
            'model_name':    model_name,
            'dataset_name':  dataset_name,
        }
        print(f'\nBest params: {gs.best_params_}')
        print(f'Best CV R²:  {gs.best_score_:.4f}')


In [ ]:
# Summary of tuning results
tuning_summary = []
for key, res in grid_results.items():
    tuning_summary.append({
        'Model': res['model_name'],
        'Dataset': res['dataset_name'],
        'Best CV R²': res['best_cv_r2'],
        'Best Params': str(res['best_params'])
    })

tuning_df = pd.DataFrame(tuning_summary).sort_values('Best CV R²', ascending=False)
print('=== Hyperparameter Tuning Summary ===')
display(tuning_df)

---
## 7. Final Model Evaluation on Test Set

We evaluate the best tuned models on the **held-out test set** (which was never seen during
cross-validation or hyperparameter tuning).

In [ ]:
# Evaluate each tuned model on its corresponding test set
final_results = []
predictions = {}  # store for plotting

for key, res in grid_results.items():
    model_name = res['model_name']
    dataset_name = res['dataset_name']
    ver = datasets_raw[dataset_name]
    
    best_model = res['grid_search'].best_estimator_
    
    # Predict on test set.
    # TransformedTargetRegressor (used for V3) back-transforms predictions
    # to the original GeV scale internally, so no manual exp() is needed.
    y_pred = best_model.predict(ver['X_test'])
    y_true = ver['y_test']

    metrics = compute_metrics(y_true, y_pred)
    metrics['Model'] = model_name
    metrics['Dataset'] = dataset_name
    metrics['Best Params'] = str(res['best_params'])
    final_results.append(metrics)
    
    predictions[key] = {
        'y_true': y_true,
        'y_pred': y_pred,
        'model_name': model_name,
        'dataset_name': dataset_name
    }
    
    print(f'{model_name} / {dataset_name}: R²={metrics["R2"]:.4f}  '
          f'RMSE={metrics["RMSE"]:.3f} GeV  MAE={metrics["MAE"]:.3f} GeV  '
          f'MAPE={metrics["MAPE"]:.2f}%')

final_df = pd.DataFrame(final_results)
print('\n=== Final Test Set Evaluation ===')
display(final_df[['Model', 'Dataset', 'R2', 'RMSE', 'MAE', 'MAPE', 'Best Params']])

### 7.1 Predicted vs Actual Scatter Plots

In [ ]:
n_plots = len(predictions)
fig, axes = plt.subplots(1, n_plots, figsize=(6 * n_plots, 5))
if n_plots == 1:
    axes = [axes]

rng = np.random.default_rng(42)
for ax, (key, pred) in zip(axes, predictions.items()):
    n   = len(pred['y_true'])
    idx = rng.choice(n, size=min(3000, n), replace=False)

    ax.scatter(pred['y_true'][idx], pred['y_pred'][idx],
               alpha=0.2, s=5, color='steelblue', rasterized=True)
    lims = [0, max(pred['y_true'].max(), pred['y_pred'].max()) * 1.05]
    ax.plot(lims, lims, 'r--', linewidth=1.5, label='Perfect fit')
    ax.set_xlabel('Actual M (GeV)', fontsize=11)
    ax.set_ylabel('Predicted M (GeV)', fontsize=11)
    m = compute_metrics(pred['y_true'], pred['y_pred'])
    ax.set_title(f"{pred['model_name']} / {pred['dataset_name']}\n"
                 f"R²={m['R2']:.4f}  RMSE={m['RMSE']:.2f} GeV", fontsize=11)
    ax.legend(fontsize=9)
    ax.set_xlim(lims)
    ax.set_ylim(lims)

plt.suptitle('Predicted vs Actual – Tuned Models (Test Set)', fontsize=14, y=1.02)
plt.tight_layout()
fig_path = OUTPUTS_DIR / 'fig_pred_vs_actual.png'
plt.savefig(fig_path, bbox_inches='tight')
plt.show()
print(f'Figure saved to {fig_path}')


### 7.2 Residual Analysis

In [ ]:
n_plots = len(predictions)
fig, axes = plt.subplots(2, n_plots, figsize=(6 * n_plots, 10))
if n_plots == 1:
    axes = axes.reshape(-1, 1)

rng = np.random.default_rng(42)
for j, (key, pred) in enumerate(predictions.items()):
    residuals = pred['y_true'] - pred['y_pred']
    idx = rng.choice(len(residuals), min(3000, len(residuals)), replace=False)

    ax = axes[0, j]
    ax.scatter(pred['y_true'][idx], residuals[idx],
               alpha=0.2, s=5, color='steelblue', rasterized=True)
    ax.axhline(0, color='red', linewidth=1, linestyle='--')
    ax.set_xlabel('Actual M (GeV)', fontsize=10)
    ax.set_ylabel('Residual (GeV)', fontsize=10)
    ax.set_title(f"{pred['model_name']} / {pred['dataset_name']}\nResiduals vs Actual", fontsize=11)

    ax = axes[1, j]
    ax.hist(residuals, bins=100, color='steelblue', edgecolor='none', alpha=0.8)
    ax.axvline(0, color='red', linewidth=1, linestyle='--')
    ax.set_xlabel('Residual (GeV)', fontsize=10)
    ax.set_ylabel('Count', fontsize=10)
    ax.set_title(f'Residual Distribution\nMean={residuals.mean():.3f}, Std={residuals.std():.3f}', fontsize=11)

plt.suptitle('Residual Analysis – Tuned Models', fontsize=14, y=1.01)
plt.tight_layout()
fig_path = OUTPUTS_DIR / 'fig_residual_analysis.png'
plt.savefig(fig_path, bbox_inches='tight')
plt.show()
print(f'Figure saved to {fig_path}')


### 7.3 Performance by Mass Region

Since the target M is multimodal (J/ψ ~3.1 GeV, Υ ~9.5 GeV, Z ~91 GeV),
it is important to evaluate how well each model performs in each resonance region.

In [ ]:
# Define mass regions based on physics
def get_mass_region(m):
    if m < 10:
        return 'Low (< 10 GeV)'
    elif m < 50:
        return 'Mid (10–50 GeV)'
    else:
        return 'High (> 50 GeV)'

region_results = []

for key, pred in predictions.items():
    regions = np.array([get_mass_region(m) for m in pred['y_true']])
    
    for region in ['Low (< 10 GeV)', 'Mid (10–50 GeV)', 'High (> 50 GeV)']:
        mask = regions == region
        if mask.sum() == 0:
            continue
        m = compute_metrics(pred['y_true'][mask], pred['y_pred'][mask])
        m['Model'] = pred['model_name']
        m['Dataset'] = pred['dataset_name']
        m['Region'] = region
        m['N_events'] = int(mask.sum())
        region_results.append(m)

region_df = pd.DataFrame(region_results)
print('=== Performance by Mass Region ===')
display(region_df[['Model', 'Dataset', 'Region', 'N_events', 'R2', 'RMSE', 'MAE', 'MAPE']]
        .style.format({'R2': '{:.4f}', 'RMSE': '{:.3f}', 'MAE': '{:.3f}', 'MAPE': '{:.2f}'}))

### 7.4 Feature Importance (Best Model)

In [ ]:
# Extract feature importance from the overall best model
best_key   = final_df.loc[final_df['R2'].idxmax()]
best_combo = f"{best_key['Model']}_{best_key['Dataset']}"
best_gs    = grid_results[best_combo]['grid_search']
best_estimator = best_gs.best_estimator_

# Unwrap TransformedTargetRegressor if present (used for V3)
est = best_estimator
if hasattr(est, 'regressor_'):
    est = est.regressor_
actual_model = est.named_steps['model'] if hasattr(est, 'named_steps') else est

if hasattr(actual_model, 'feature_importances_'):
    feat_names  = datasets_raw[best_key['Dataset']]['feature_names']
    importances = actual_model.feature_importances_
    feat_imp    = pd.Series(importances, index=feat_names).sort_values(ascending=False)

    fig, ax = plt.subplots(figsize=(10, 6))
    feat_imp.head(15).plot(kind='barh', ax=ax, color='steelblue', edgecolor='none')
    ax.invert_yaxis()
    ax.set_xlabel('Feature Importance (MDI)', fontsize=12)
    ax.set_title(f"Feature Importances – Best Model: {best_key['Model']} / {best_key['Dataset']}",
                 fontsize=13)
    plt.tight_layout()
    fig_path = OUTPUTS_DIR / 'fig_feature_importance_best.png'
    plt.savefig(fig_path, bbox_inches='tight')
    plt.show()
    print(f'Figure saved to {fig_path}')
    print('\nTop 10 features:')
    print(feat_imp.head(10).to_string())
else:
    print(f'Feature importances not available for {type(actual_model).__name__}')


---
## 8. Comparison of Default vs Tuned Performance

In [ ]:
# Compare default CV R² with tuned CV R² for selected combinations
comparison_rows = []

for key, res in grid_results.items():
    model_name = res['model_name']
    dataset_name = res['dataset_name']
    
    # Find default R² from screening
    default_r2 = cv_df[
        (cv_df['Model'] == model_name) & (cv_df['Dataset'] == dataset_name)
    ]['R2_mean'].values[0]
    
    tuned_r2 = res['best_cv_r2']
    
    comparison_rows.append({
        'Model': model_name,
        'Dataset': dataset_name,
        'Default R² (CV)': default_r2,
        'Tuned R² (CV)': tuned_r2,
        'Improvement': tuned_r2 - default_r2
    })

comp_df = pd.DataFrame(comparison_rows)
print('=== Default vs Tuned Performance ===')
display(comp_df.style.format({
    'Default R² (CV)': '{:.4f}',
    'Tuned R² (CV)': '{:.4f}',
    'Improvement': '{:+.4f}'
}))

---
## 9. Discussion

### 9.1 Key Findings

*(Fill in after running the experiments. Template below.)*

- **Best performing model:** [Model] on dataset [V?] achieved R² = [?], RMSE = [?] GeV.
- **Effect of preprocessing:** Log-transformation of energy features [improved/did not improve] performance for [models]. Engineered physics features (V4) [helped/did not help] because...
- **Scaling insight:** As noted in the Deliverable 2 feedback, removing unnecessary scaling for tree-based models [had no effect / slightly improved training speed].
- **Mass region performance:** Models performed best in the [high/low/mid] mass region because...

### 9.2 Comparison with Literature

| Aspect | Kilic et al. (2023) | Our Study |
|--------|--------------------|-----------|
| Dataset | CERN dielectron (same) | CERN dielectron (same) |
| Target | Z boson mass | Invariant mass M (full range) |
| Best Algorithm | [?] | [?] |
| Features Used | [?] | Original + engineered physics features |
| Best R² | [?] | [?] |
| Best RMSE | [?] | [?] GeV |

### 9.3 Domain-Specific Validation

*(Discuss: Do the model's predictions make physical sense? Does it correctly identify resonance peaks? Are the most important features physically meaningful?)*

### 9.4 Study Limitations

- SVR training required subsampling to 20K events due to O(n²) complexity
- The multimodal target distribution (3 resonance peaks) makes it challenging for models optimising MSE
- [Add more based on your findings]

---
## 10. Final Summary

In [ ]:
print('=' * 75)
print('DELIVERABLE 3 – FINAL SUMMARY')
print('=' * 75)
print()
print('Dataset: CERN Electron Collision – 100,000 dielectron events')
print('Task:    Regression – predict invariant mass M (GeV)')
print()
print(f'Models evaluated:      KNN, SVR, Random Forest, XGBoost')
print(f'Dataset versions:      V1 (baseline), V2 (log features), V3 (log target), V4 (engineered)')
print(f'Models tuned:          {SELECTED_MODELS}')
print(f'Datasets tuned on:     {SELECTED_DATASETS}')
print()
print('--- Best Model (Test Set) ---')
best_row = final_df.loc[final_df['R2'].idxmax()]
print(f'Model:     {best_row["Model"]}')
print(f'Dataset:   {best_row["Dataset"]}')
print(f'R²:        {best_row["R2"]:.4f}')
print(f'RMSE:      {best_row["RMSE"]:.3f} GeV')
print(f'MAE:       {best_row["MAE"]:.3f} GeV')
print(f'MAPE:      {best_row["MAPE"]:.2f}%')
print(f'Params:    {best_row["Best Params"]}')
print()
print('=' * 75)

---
## References

[1] McCauley, T. (2014). Events with two electrons from 2010. CERN Open Data Portal. DOI: 10.7483/OPENDATA.CMS.PCSW.AHVG  
[2] Kilic, H., Ozturk, S., & Yildirim, E. (2023). Machine learning model performances for the Z boson mass identification through dielectron decay channel. *The European Physical Journal Plus*, 138(1), 87.  
[3] Pedregosa, F., et al. (2011). Scikit-learn: Machine Learning in Python. *JMLR*, 12, 2825-2830.  
[4] Chen, T., & Guestrin, C. (2016). XGBoost: A Scalable Tree Boosting System. *KDD 2016*.  
[5] Breiman, L. (2001). Random Forests. *Machine Learning*, 45(1), 5-32.  
[6] Vapnik, V. (1995). *The Nature of Statistical Learning Theory*. Springer.  
[7] Cover, T., & Hart, P. (1967). Nearest neighbor pattern classification. *IEEE Trans. Information Theory*, 13(1), 21-27.